<div id='top'>包括以下操作：</div>
<li><a href='#1'>BCELoss</a></li>
<li><a href='#2'>定位损失</a></li>
<li><a href='#3'>传统损失计算：L1/L2</a></li>
<li><a href='#4'>IoU损失</a></li>
<li><a href='#5'>GIoU损失</a></li>
<li><a href='#6'>DIoU损失</a></li>
<li><a href='#7'>CIoU损失</a></li>

[IOU、GIOU、DIOU、CIOU损失函数详解](https://zhuanlan.zhihu.com/p/359982543)

<div style='color:skyblue; font-size:24px' id='1'>BCEloss 计算(二元交叉熵损失)</div>

<a href='#top'>▲ Top</a>

（置信度损失）

在目标检测模型（如YOLOv3、YOLOv8）中，类别损失通常采用二元交叉熵（BCE），将每个类别的预测视为独立的二分类问题。因此每个类别的预测值相加不等于 1.

In [2]:
# 类别损失计算
import torch
import numpy as np

# 法一：使用官方的计算
loss=torch.nn.BCELoss(reduction='none') # reduction='none' 不设置的话，就会对结果取均值
# 假设一张图片上有两个物体，可以放进3个类别里。下面就是针对这两个物体，在3个类别上分别取得的概率。
po=[[0.1,0.8,0.9],[0.2,0.7,0.8]]
p = torch.tensor(po, requires_grad=True)  # 真实标签

go=[[0.,0.,1.],[0.,0.,1.]]
g=torch.tensor(go)

l = loss(input=p, target=g)  
print(np.round(l.detach().numpy(), 5)) # l.detach().numpy()：去除梯度信息，转化成 numpy格式，对每个值进行四舍五入，只保留5位小数

# 法二：自定义的计算
def bce(c, o):
    return np.round(-(o*np.log(c) + (1-o)*np.log(1-c)), 5)

c = np.array(po)
o = np.array(go)

print(bce(c, o))

[[0.10536 1.60944 0.10536]
 [0.22314 1.20397 0.22314]]
[[0.10536 1.60944 0.10536]
 [0.22314 1.20397 0.22314]]


<div style='color:skyblue; font-size:24px' id='2'>定位损失</div>

<a href='#top'>▲ Top</a>

In [ ]:
yolov3使用的 sum of squared error loss
但是后续如yolov3 SPP 使用的 CIoU loss

<div style='color:skyblue; font-size:24px' id='3'>传统损失计算：L1/L2</div>

<a href='#top'>▲ Top</a>

<div style='color:skyblue; font-size:24px' id='4'>IoU损失</div>

<a href='#top'>▲ Top</a>

具有尺寸不变性；IoU Loss的尺度不变性是指其损失值不会因目标物体的实际尺寸（如边界框的绝对宽度和高度）不同而产生变化，而是仅依赖于预测框与真实框的相对位置和重叠比例。

与传统的L1/L2 Loss直接计算坐标值的绝对差异（如中心点坐标、宽高的差值），而目标的绝对尺寸越大，其坐标差值可能越大，导致损失值随目标尺度增大而显著增加
。例如，预测一个边长为10的框与边长为5的框的误差（如差值为5）与预测边长为100的框与边长为50的框的误差（差值同样为5）在L1/L2 Loss中会被视为相同，但实际IoU差异可能完全不同。这种特性会导致模型对不同尺度的目标学习不平衡。

后续的改进方法（如GIoU、DIoU、CIoU）在保留尺度不变性的基础上，通过引入中心点距离、长宽比等附加信息进一步优化了回归过程。

<div style='color:skyblue; font-size:24px' id='5'>GIoU损失</div>

<a href='#top'>▲ Top</a>

IoU Loss对距离不敏感的原因在于其仅通过交集与并集的比值衡量重叠程度，当预测框与真实框无重叠时，IoU值为0且梯度消失，无法反映两者的距离差异。例如，两个相距较远的框与两个略微偏移的框在IoU Loss中可能表现相同（IoU均为0），导致模型无法学习如何调整位置。

GIoU Loss通过引入最小外接矩形（即同时包含预测框和真实框的最小闭合区域）解决了这一问题。

在当两个矩形框距离很远时，并集u趋近于0的意思是：因为并集u始终是保持不变的，当两者的距离无穷远时，u还是保持的话，就会显得和0差不多了。

<div style='color:orange; font-size:18px'>并集面积的物理意义</div>

无论两框距离多远，只要框的尺寸固定，u = Area(A) + Area(B) 始终保持不变。例如，两框均为1x1的正方形，则 u = 1 + 1。
u 不会因距离变化而趋向无穷大。而外接矩形面积会因为距离变化而趋向无穷大。

<div style='color:orange; font-size:18px'>GIOU缺点</div>
在某些情况下，GIOU就退化为IOU。

此外，GIOU和IOU还有两个缺点：收敛较慢、回归不够准确。

<div style='color:skyblue; font-size:24px' id='6'>DIoU损失</div>

<a href='#top'>▲ Top</a>

加强收敛、达到更高的定位精度。

DIoU能直接最小化两个box之间的距离，因此收敛速度更快。


<div style='color:orange; font-size:18px'>收敛速度优势</div>

<li>对比GIoU的缺陷：当预测框与真实框无重叠时，GIoU需先扩大预测框使其与真实框重叠，再逐步调整形状，导致收敛缓慢。而DIoU直接通过中心点距离提供优化方向，无需先扩大框的冗余步骤。</li>
<li>包含或平行情况：当预测框被真实框完全包含或两框处于水平/垂直方向时，GIoU的惩罚项失效（退化为IoU），而DIoU仍能通过中心距离有效优化。</li>
<li>尺度不变性：归一化项（除以对角线距离的平方）消除了目标尺寸对距离度量的影响，确保不同大小的框都能高效优化。</li>

<div style='color:skyblue; font-size:24px' id='7'>CIoU损失</div>

<a href='#top'>▲ Top</a>

进一步引入了长宽比相似性作为优化项。